# Mutual Fund Performance Analytics & Scorecard

This notebook contains the complete performance analysis, risk metrics, regression analysis against benchmarks, drawdown calculation, and a composite scorecard for 40 mutual fund schemes.

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["font.size"] = 12

## 1. Data Loading and Preprocessing
We load the datasets: fund master metadata, NAV histories, and benchmark index values.

In [ ]:
df_funds = pd.read_csv("data/processed/fund_master.csv")
df_nav = pd.read_csv("data/processed/nav_history.csv")
df_bench = pd.read_csv("data/processed/benchmark_indices.csv")

# Parse dates
df_nav['date'] = pd.to_datetime(df_nav['date'])
df_bench['date'] = pd.to_datetime(df_bench['date'])

# Sort NAV history by fund and date
df_nav = df_nav.sort_values(by=['amfi_code', 'date']).reset_index(drop=True)

print(f"Loaded {df_funds['amfi_code'].nunique()} funds and {len(df_nav)} NAV history rows.")

## 2. Daily Returns Calculation and Distribution Validation
We compute the daily returns for each scheme using the formula:
$$daily\_return = \frac{nav_t}{nav_{t-1}} - 1$$
We also plot the distribution of daily returns to ensure there are no extreme outliers or data corruption anomalies.

In [ ]:
df_nav['daily_return'] = df_nav.groupby('amfi_code')['nav'].pct_change()

# Distribution description
print("Daily Returns Distribution Summary:")
print(df_nav['daily_return'].describe())

# Plot distribution histogram & KDE
plt.figure(figsize=(10, 6))
sns.histplot(df_nav['daily_return'].dropna(), bins=100, kde=True, color='royalblue')
plt.title("Distribution of Daily Returns Across All 40 Schemes")
plt.xlabel("Daily Return")
plt.ylabel("Frequency")
plt.xlim(-0.06, 0.06)
plt.tight_layout()
plt.show()

## 3. CAGR (Compound Annual Growth Rate) Calculation
We calculate the annualized CAGR for 1-year, 3-year, and 5-year periods using:
$$CAGR = \left(\frac{NAV_{end}}{NAV_{start}}\right)^{\frac{1}{n}} - 1$$
Where:
- $NAV_{end}$ is the latest NAV.
- $NAV_{start}$ is the NAV at the start of the period (closest matching date in the historical series).
- $n$ is the actual number of years between start and end date: $\frac{\text{days}}{365.25}$.
- *Note:* Since the NAV history spans approximately 4.4 years, the 5-year CAGR cannot be calculated and is marked as `NaN`.

In [ ]:
# Setup results list
results = []
rf_annual = 0.065
rf_daily = rf_annual / 252

unique_funds = df_funds['amfi_code'].unique()

for amfi_code in unique_funds:
    fund_info = df_funds[df_funds['amfi_code'] == amfi_code].iloc[0]
    fund_name = fund_info['scheme_name']
    expense_ratio = fund_info['expense_ratio_pct']
    
    # Filter and sort
    df_f = df_nav[df_nav['amfi_code'] == amfi_code].sort_values(by='date').copy()
    if len(df_f) < 2:
        continue
        
    date_end = df_f['date'].max()
    nav_end = df_f.loc[df_f['date'] == date_end, 'nav'].values[0]
    
    # 1-Year CAGR
    date_1yr_target = date_end - pd.DateOffset(years=1)
    idx_1yr = (df_f['date'] - date_1yr_target).abs().idxmin()
    row_1yr = df_f.loc[idx_1yr]
    trading_days_1yr = df_f[(df_f['date'] > row_1yr['date']) & (df_f['date'] <= date_end)].shape[0]
    cagr_1yr = (nav_end / row_1yr['nav']) ** (252 / trading_days_1yr) - 1 if trading_days_1yr > 0 else np.nan
    
    # 3-Year CAGR
    date_3yr_target = date_end - pd.DateOffset(years=3)
    idx_3yr = (df_f['date'] - date_3yr_target).abs().idxmin()
    row_3yr = df_f.loc[idx_3yr]
    trading_days_3yr = df_f[(df_f['date'] > row_3yr['date']) & (df_f['date'] <= date_end)].shape[0]
    cagr_3yr = (nav_end / row_3yr['nav']) ** (252 / trading_days_3yr) - 1 if trading_days_3yr > 0 else np.nan
    
    # 5-Year CAGR (not enough history)
    cagr_5yr = np.nan
    
    # Ratios (Sharpe & Sortino)
    returns = df_f['daily_return'].dropna()
    if len(returns) > 0:
        mean_ret = returns.mean()
        std_ret = returns.std()
        sharpe = (mean_ret - rf_daily) / std_ret * np.sqrt(252) if std_ret > 0 else np.nan
        
        downside_returns = returns[returns < 0]
        downside_std = downside_returns.std()
        sortino = (mean_ret - rf_daily) / downside_std * np.sqrt(252) if downside_std > 0 else np.nan
    else:
        sharpe = np.nan
        sortino = np.nan
        
    # Alpha & Beta against NIFTY100
    df_fund_ret = df_f[['date', 'daily_return']].dropna()
    df_bench_piv = df_bench.pivot(index='date', columns='index_name', values='close_value')
    df_bench_returns = df_bench_piv.pct_change()
    df_bench_ret_f = df_bench_returns[['NIFTY100']].dropna().reset_index()
    
    df_merged_ret = pd.merge(df_fund_ret, df_bench_ret_f, on='date', how='inner')
    if len(df_merged_ret) > 10:
        slope, intercept, r_val, p_val, std_err = stats.linregress(
            df_merged_ret['NIFTY100'], df_merged_ret['daily_return']
        )
        beta = slope
        alpha = intercept * 252
    else:
        beta = np.nan
        alpha = np.nan
        
    # Max Drawdown
    df_f['running_max'] = df_f['nav'].cummax()
    df_f['drawdown'] = df_f['nav'] / df_f['running_max'] - 1
    max_dd = df_f['drawdown'].min()
    
    # Worst drawdown date range
    trough_idx = df_f['drawdown'].idxmin()
    trough_row = df_f.loc[trough_idx]
    trough_date = trough_row['date']
    
    # Peak date
    running_max_val = trough_row['running_max']
    peak_row = df_f[(df_f['date'] <= trough_date) & (df_f['nav'] == running_max_val)]
    if len(peak_row) > 0:
        peak_date = peak_row['date'].iloc[0]
    else:
        peak_idx = df_f.loc[:trough_idx, 'nav'].idxmax()
        peak_date = df_f.loc[peak_idx, 'date']
        
    # Tracking Error over 3 years
    df_3yr = df_f[(df_f['date'] >= '2023-05-29') & (df_f['date'] <= '2026-05-29')].copy()
    df_bench_3yr = df_bench_returns[(df_bench_returns.index >= '2023-05-29') & (df_bench_returns.index <= '2026-05-29')].reset_index()
    df_te_merge = pd.merge(df_3yr[['date', 'daily_return']], df_bench_3yr, on='date', how='inner')
    
    if len(df_te_merge) > 10:
        te_nifty50 = (df_te_merge['daily_return'] - df_te_merge['NIFTY50']).std() * np.sqrt(252)
        te_nifty100 = (df_te_merge['daily_return'] - df_te_merge['NIFTY100']).std() * np.sqrt(252)
    else:
        te_nifty50 = np.nan
        te_nifty100 = np.nan
        
    results.append({
        'amfi_code': amfi_code,
        'scheme_name': fund_name,
        'expense_ratio_pct': expense_ratio,
        'cagr_1yr': cagr_1yr,
        'cagr_3yr': cagr_3yr,
        'cagr_5yr': cagr_5yr,
        'sharpe_ratio': sharpe,
        'sortino_ratio': sortino,
        'alpha': alpha,
        'beta': beta,
        'max_drawdown': max_dd,
        'worst_dd_peak_date': peak_date.strftime('%Y-%m-%d'),
        'worst_dd_trough_date': trough_date.strftime('%Y-%m-%d'),
        'tracking_error_nifty50': te_nifty50,
        'tracking_error_nifty100': te_nifty100
    })

df_results = pd.DataFrame(results)
print("CAGR comparison table for first 5 funds:")
print(df_results[['scheme_name', 'cagr_1yr', 'cagr_3yr', 'cagr_5yr']].head())

## 4. Sharpe and Sortino Ratios Ranking
The **Sharpe Ratio** measures the excess return per unit of total risk:
$$Sharpe = \frac{R_p - R_f}{\sigma_p}$$
The **Sortino Ratio** evaluates risk-adjusted return by using only downside deviation (volatility of negative daily returns) to avoid penalizing positive volatility:
$$Sortino = \frac{R_p - R_f}{\sigma_d}$$
We rank all 40 funds by Sharpe ratio.

In [ ]:
df_sharpe_rank = df_results[['scheme_name', 'sharpe_ratio', 'sortino_ratio']].sort_values(by='sharpe_ratio', ascending=False).reset_index(drop=True)
df_sharpe_rank['Sharpe Rank'] = df_sharpe_rank.index + 1
print("Top 10 Funds Ranked by Sharpe Ratio:")
print(df_sharpe_rank.head(10))

## 5. Alpha and Beta against Nifty 100
We run OLS linear regression of daily fund returns on Nifty 100 daily returns:
$$R_{\text{fund}} = \alpha + \beta \cdot R_{\text{Nifty100}} + \epsilon$$
- **Beta ($\beta$)** measures sensitivity to the Nifty 100 index.
- **Alpha ($\alpha$)** represents risk-adjusted outperformance (annualized by multiplying the intercept by 252).

In [ ]:
df_alpha_beta_res = df_results[['amfi_code', 'scheme_name', 'alpha', 'beta']].sort_values(by='alpha', ascending=False).reset_index(drop=True)
df_alpha_beta_res.to_csv("alpha_beta.csv", index=False)
print("Top 5 Funds by Alpha against Nifty 100:")
print(df_alpha_beta_res.head())

## 6. Maximum Drawdown & Worst Drawdown Date Range
Maximum Drawdown (MDD) is the maximum loss from peak to trough:
$$MDD = \min\left(\frac{NAV_t}{\max_{i \le t} NAV_i} - 1\right)$$
We find the worst drawdown and identify its peak-to-trough date range for each fund.

In [ ]:
df_dd_res = df_results[['scheme_name', 'max_drawdown', 'worst_dd_peak_date', 'worst_dd_trough_date']].sort_values(by='max_drawdown').reset_index(drop=True)
print("Worst 5 Funds by Maximum Drawdown:")
print(df_dd_res.head())

## 7. Composite Fund Scorecard
We build a composite score (0–100) using percentile ranks:
$$\text{Score} = 30\% \cdot \text{Rank}_{\text{3yr Return}} + 25\% \cdot \text{Rank}_{\text{Sharpe}} + 20\% \cdot \text{Rank}_{\text{Alpha}} + 15\% \cdot \text{Rank}_{\text{Expense Ratio (inverse)}} + 10\% \cdot \text{Rank}_{\text{Max DD (inverse)}}$$
Ranks are converted to percentiles (0 to 100). Higher score represents a better combination of return, risk-adjusted performance, low fees, and downside protection.

In [ ]:
df_results['rank_3yr_ret'] = df_results['cagr_3yr'].rank(pct=True) * 100
df_results['rank_sharpe'] = df_results['sharpe_ratio'].rank(pct=True) * 100
df_results['rank_alpha'] = df_results['alpha'].rank(pct=True) * 100
df_results['rank_expense'] = df_results['expense_ratio_pct'].rank(ascending=False, pct=True) * 100
df_results['rank_max_dd'] = df_results['max_drawdown'].rank(pct=True) * 100

df_results['scorecard_score'] = (
    0.30 * df_results['rank_3yr_ret'] +
    0.25 * df_results['rank_sharpe'] +
    0.20 * df_results['rank_alpha'] +
    0.15 * df_results['rank_expense'] +
    0.10 * df_results['rank_max_dd']
)

df_results = df_results.sort_values(by='scorecard_score', ascending=False).reset_index(drop=True)
df_results['rank'] = df_results.index + 1

df_scorecard_out = df_results[[
    'rank', 'amfi_code', 'scheme_name', 'scorecard_score', 
    'cagr_1yr', 'cagr_3yr', 'cagr_5yr', 'sharpe_ratio', 'sortino_ratio', 
    'max_drawdown', 'worst_dd_peak_date', 'worst_dd_trough_date', 'expense_ratio_pct'
]]
df_scorecard_out.to_csv("fund_scorecard.csv", index=False)
print("Top 10 Funds by Scorecard Score:")
print(df_results[['rank', 'scheme_name', 'scorecard_score', 'cagr_3yr', 'sharpe_ratio', 'alpha']].head(10))

## 8. Benchmark Comparison Chart and Tracking Error
We select the top 5 funds from the scorecard and plot their cumulative performance over the last 3 years (`2023-05-29` to `2026-05-29`) alongside Nifty 50 and Nifty 100.
We also compute the annualized tracking error:
$$\text{Tracking Error} = \sigma(R_{\text{fund}} - R_{\text{benchmark}}) \cdot \sqrt{252}$$
All plots and CSVs are saved to the workspace root directory.

In [ ]:
top_5_codes = df_results['amfi_code'].head(5).tolist()
df_nav_3yr = df_nav[(df_nav['date'] >= '2023-05-29') & (df_nav['date'] <= '2026-05-29')].copy()

plt.figure(figsize=(12, 7))

# Plot funds
for code in top_5_codes:
    fund_name = df_results[df_results['amfi_code'] == code]['scheme_name'].values[0]
    df_f = df_nav_3yr[df_nav_3yr['amfi_code'] == code].sort_values(by='date').copy()
    if len(df_f) > 0:
        first_nav = df_f['nav'].iloc[0]
        df_f['indexed_nav'] = (df_f['nav'] / first_nav) * 100
        plt.plot(df_f['date'], df_f['indexed_nav'], label=fund_name, linewidth=1.5)

# Plot Benchmarks
df_bench_3yr = df_bench[(df_bench['date'] >= '2023-05-29') & (df_bench['date'] <= '2026-05-29')].copy()
for bench_name in ['NIFTY50', 'NIFTY100']:
    df_b = df_bench_3yr[df_bench_3yr['index_name'] == bench_name].sort_values(by='date').copy()
    if len(df_b) > 0:
        first_val = df_b['close_value'].iloc[0]
        df_b['indexed_val'] = (df_b['close_value'] / first_val) * 100
        linestyle = '--' if bench_name == 'NIFTY50' else '-.'
        plt.plot(df_b['date'], df_b['indexed_val'], label=bench_name, linewidth=2.5, linestyle=linestyle, color='black' if bench_name == 'NIFTY100' else 'darkorange')

plt.title("Top 5 Mutual Funds vs Benchmarks (3-Year Cumulative Returns)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Indexed Value (Base 100)", fontsize=12)
plt.legend(loc='upper left', frameon=True, facecolor='white', edgecolor='lightgray')
plt.tight_layout()
plt.savefig("benchmark_comparison.png", dpi=300)
plt.show()

# Show Tracking Error for top 5 funds
print("Tracking Error for Top 5 Funds:")
print(df_results[['scheme_name', 'tracking_error_nifty50', 'tracking_error_nifty100']].head(5))